# Hypothesis Testing — ANOVA

This notebook performs a one-way ANOVA test to verify whether the three K-Means clusters
identified in notebook 04 have statistically different mean accident counts.

If the ANOVA confirms significant differences between the clusters, it validates our segmentation —
meaning the clusters represent genuinely distinct groups rather than random groupings.

Null hypothesis (H0): The mean number of road accidents is the same across all three clusters.
Alternative hypothesis (H1): At least one cluster has a significantly different mean accident count.

In [19]:
# import libraries
import pandas as pd
import numpy as np
from scipy import stats
import sys
sys.path.append('../config')
from config import *
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [20]:
# import clustered_istat_data from clean
df_merged_clustered_istat_data=pd.read_csv(CLEAN_PATH + 'clustered_istat_data.csv')

# display the unique cluster labels

df_merged_clustered_istat_data['cluster_label'].unique()

array(['Low Risk', 'High Per Capita', 'High Per Km2'], dtype=object)

In [21]:
# separate ROADACC values by cluster
low_risk = df_merged_clustered_istat_data[df_merged_clustered_istat_data['cluster_label'] == 'Low Risk']['ROADACC']
high_per_capita=df_merged_clustered_istat_data[df_merged_clustered_istat_data['cluster_label']== 'High Per Capita']['ROADACC']
high_per_km2=df_merged_clustered_istat_data[df_merged_clustered_istat_data['cluster_label']=='High Per Km2']['ROADACC']

# perform ANOVA test
f_stat, p_value = stats.f_oneway(low_risk, high_per_capita, high_per_km2)
print(f"F-statistic: {f_stat}")
print(f"P-value: {p_value}")


F-statistic: 9150.326563937266
P-value: 0.0


## ANOVA Test Results

The one-way ANOVA test returns an F-statistic of 9150.33 and a p-value of 0.0.

The F-statistic is extremely high, indicating that the variance between the three clusters
is far greater than the variance within each cluster.

The p-value is effectively zero, well below the standard threshold of 0.05.
We therefore reject the null hypothesis (H0) and confirm the alternative hypothesis (H1):
the three clusters have statistically different mean accident counts.

This result validates our K-Means segmentation — the clusters represent genuinely distinct
groups rather than random groupings, and the differences between them are not due to chance.

In [22]:
# perfrom Tukey's HSD test
tukey = pairwise_tukeyhsd(
    endog=df_merged_clustered_istat_data['ROADACC'],
    groups=df_merged_clustered_istat_data['cluster_label'],
    alpha=0.05
)
print(tukey)

          Multiple Comparison of Means - Tukey HSD, FWER=0.05          
     group1        group2     meandiff p-adj   lower     upper   reject
-----------------------------------------------------------------------
High Per Capita High Per Km2  583.8731   0.0   572.511  595.2351   True
High Per Capita     Low Risk  -45.0903   0.0  -48.4024  -41.7783   True
   High Per Km2     Low Risk -628.9634   0.0 -640.0342 -617.8926   True
-----------------------------------------------------------------------


## Tukey HSD Post-Hoc Test Results

The Tukey HSD test confirms that all three clusters are statistically different from each other
(all p-values = 0.0, all reject = True).

Ranking by mean accident count (highest to lowest):
1. High Per Km2 — highest absolute accident count, 628 more accidents on average than Low Risk
2. High Per Capita — intermediate, 45 more accidents on average than Low Risk  
3. Low Risk — lowest absolute accident count

This validates our K-Means clustering: the three groups represent genuinely distinct risk profiles,
not random groupings. The High Per Km2 cluster (densely urbanized municipalities like Milano)
consistently shows the highest accident counts, making them the priority targets for
road safety infrastructure investments.